In [ ]:
import importlib
import functions

importlib.reload(functions)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from functions import *

In [ ]:
OWL_PATH1 = "data/oeo-full.owl"
OWL_PATH2 = "data/go-plus.owl"

In [ ]:
base_cfg = BoxConfig(
    dim = 6,
    steps = 100000,
    seed = 42,
    size_weight = 0.1,
    subclass_weight = 2.0,
    disjoint_weight = 1.0,
    big_box_weight = 0.1,
    depth_scale = 0.5,
    distance_weight = 0.1)

In [ ]:
schedule=CurriculumSchedule(
    subclass_start = 0.0,   # structural foundation first
    disjoint_start = 0.0,   # separation only after containment exists
    sibling_start = 0.7,
    big_box_start = 0.7,   # size control
    ramp = False
)

In [ ]:
schedule_go=CurriculumSchedule(
    subclass_start = 0.1,   # structural foundation first
    disjoint_start = 0.0,   # separation only after containment exists
    sibling_start = 0.7,
    big_box_start = 0.9,   # size control
    ramp = False
)

In [ ]:
results_plain_oeo = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_from_owl,
    dims = range(2, 21),
    cfg = base_cfg,
    path = "saved/plain_oeo",
)


In [ ]:
results_curriculum_oeo = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(2, 21),
    cfg = base_cfg,
    path = "saved/curr_oeo",
)

In [ ]:
results_plain_oeo_noise = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_from_owl,
    dims = range(2, 21),
    cfg = base_cfg,
    noise = True,
    path = "saved/plain_oeo_noise",
)

In [ ]:
results_curriculum_oeo_noise = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(2, 21),
    cfg = base_cfg,
    noise = True,
    path = "saved/curr_oeo_noise",
)

In [ ]:
results_plain_go = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_from_owl,
    dims = range(5, 21),
    cfg = base_cfg,
    path = "saved/plain_go",
    steps = 10000
)

In [ ]:
results_curriculum_go_new = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule_go,
    dims = range(5, 10),
    cfg = base_cfg,
    path = "saved/curr_go_new",
    steps = 15000
)

careful how to call it, must use load_owl or load_owl_with errors accordingly!

In [ ]:
classes, subclass_of, disjoint_pairs = load_owl(OWL_PATH1)
results_plain_oeo = load_sweep_results("saved/plain_oeo", classes, subclass_of, disjoint_pairs)
results_curriculum_oeo = load_sweep_results("saved/curr_oeo", classes, subclass_of, disjoint_pairs)
#

In [ ]:
classes, subclass_of, disjoint_pairs = load_owl_with_errors(OWL_PATH1)
results_plain_oeo_noise = load_sweep_results("saved/plain_oeo_noise", classes, subclass_of, disjoint_pairs)
results_curriculum_oeo_noise = load_sweep_results("saved/curr_oeo_noise", classes, subclass_of, disjoint_pairs)

In [ ]:
classes, subclass_of, disjoint_pairs = load_owl(OWL_PATH2)
results_plain_go = load_sweep_results("saved/plain_go", classes, subclass_of, disjoint_pairs)
#results_curriculum_go = load_sweep_results("saved/curr_go", classes, subclass_of, disjoint_pairs)

### eval

In [ ]:
plot_sweep_comparison(results_plain_oeo, results_curriculum_oeo,"OEO")

In [ ]:
plot_sweep_comparison(results_plain_oeo_noise, results_curriculum_oeo_noise, "OEO Noise")

In [ ]:
plot_sweep_comparison(results_plain_go, results_curriculum_go_new,"GO")

In [ ]:
eval_plain_oeo = evaluate_models(results_plain_oeo)
plot_evaluation(eval_plain_oeo, title="Plain Box Embedding — OEO", save_path="eval_oeo_plain.eps", fmt="eps", legend=False)

eval_curriculum_oeo = evaluate_models(results_curriculum_oeo)
plot_evaluation(eval_curriculum_oeo, title="Curriculum Box Embedding — OEO", save_path="eval_oeo_curr.eps", fmt="eps")


In [ ]:
eval_plain_oeo_noise = evaluate_models(results_plain_oeo_noise)
plot_evaluation(eval_plain_oeo_noise, title="Plain Box Embedding — OEO Noise", save_path="eval_oeo_plain_niose.eps", fmt="eps", legend=False)

eval_curriculum_oeo_noise = evaluate_models(results_curriculum_oeo_noise)
plot_evaluation(eval_curriculum_oeo_noise, title="Curriculum Box Embedding — OEO Noise", save_path="eval_oeo_curr_noise.eps", fmt="eps")


In [ ]:
eval_plain_go = evaluate_models(results_plain_go)
plot_evaluation(eval_plain_go, title="Plain Box Embedding — GO", save_path="eval_go_plain.eps", fmt="eps", legend=True)


In [ ]:
table_sweep_comparison(results_plain_oeo, results_curriculum_oeo,"OEO")

In [ ]:
table_sweep_comparison(results_plain_oeo_noise, results_curriculum_oeo_noise,"OEO")

In [ ]:
table_sweep_comparison(results_plain_go, results_curriculum_go_new,"GO")

In [ ]:
eval_conc = evaluate_concluded_relationships(results_plain_oeo, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc, "OEO – Concluded Relationships (Plain)")

In [ ]:
eval_conc_go = evaluate_concluded_relationships(results_plain_go, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc_go, "GO – Concluded Relationships (Plain)")

In [ ]:
eval_conc_go_curr = evaluate_concluded_relationships(results_curriculum_go, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc_go_curr, "GO – Concluded Relationships (Curr)")

In [ ]:
eval_conc = evaluate_concluded_relationships(results_plain_oeo, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc, "OEO – Concluded Relationships (Plain - Noise)")

In [ ]:
eval_conc_curr = evaluate_concluded_relationships(results_curriculum_oeo, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc_curr, "OEO – Concluded Relationships (Curriculum)")

In [ ]:
eval_conc_curr = evaluate_concluded_relationships(results_curriculum_oeo_noise, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc_curr, "OEO – Concluded Relationships (Curriculum Noise)")

In [ ]:
results = sweep_schedule_combinations(
    owl_path=OWL_PATH1,
    params={
        "disjoint_start" : [0.0, 0.5],
        "subclass_start" : [0.0, 0.5],
        "big_box_start"  : [0.0, 0.5],
        "sibling_start"  : [0.0, 0.5],
    },
    dim=4,
    steps=10000,
    base_schedule=CurriculumSchedule(ramp=False),
)

In [ ]:
plot_combo_heatmap_unified2(results, figsize=(10, 5))

In [ ]:
results = sweep_schedule_combinations(
    owl_path=OWL_PATH2,
    params={
        "disjoint_start" : [0.0, 0.5],
        "subclass_start" : [0.0, 0.5],
        "big_box_start"  : [0.0, 0.5],
        "sibling_start"  : [0.0, 0.5],
    },
    dim=4,
    steps=1000,
    base_schedule=CurriculumSchedule(ramp=False),
)

In [ ]:
results2 = sweep_schedule_combinations(
    owl_path=OWL_PATH2,
    params={
        "disjoint_start" : [0.0, 0.1],
        "subclass_start" : [0.0],
        "big_box_start"  : [0.7, 0.9],
        "sibling_start"  : [0.5, 0.7],
    },
    dim=8,
    steps=10000,
    base_schedule=CurriculumSchedule(ramp=False),
)

In [ ]:
plot_combo_heatmap_unified2(results, figsize=(10, 5))

In [ ]:
results2 = sweep_schedule_combinations(
    owl_path=OWL_PATH2,
    params={
        "disjoint_start" : [0.0, 0.1],
        "subclass_start" : [0.0, 0.1],
        "big_box_start"  : [0.7, 0.9],
        "sibling_start"  : [0.5, 0.7],
    },
    dim=8,
    steps=10000,
    base_schedule=CurriculumSchedule(ramp=False),
)

In [ ]:
plot_combo_heatmap_unified2(results2, figsize=(10, 5))